# 2.2 Generate AEDP Different-Aggregation Datasets

Generate 12 weather-enhanced PyNNLF datasets from the valid per-household checkpoints produced by notebook `2.1`.

Important modelling choice: this notebook intentionally forward-fills each household's 30-minute checkpoint before aggregating households. This does not exactly reproduce the historical `ds10`/`ds11` group-level aggregation order, but it gives each sampled household/group a complete profile before summation, which is the desired behaviour for this aggregation-level experiment.


## 1. Setup And Paths

In [1]:
from pathlib import Path
import sys


def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")


PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")

import pandas as pd
import numpy as np

PROCESSED_DIR = Path(r"C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\2. processed")
CLEANED_WORKSPACE = Path(r"C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\3. cleaned\AEDP_different_aggregation")
SITE_LIST_PATH = PROCESSED_DIR / "aedp_cluster_2_2_3years.xlsx"
WEATHER_PATH = PROCESSED_DIR / "aedp_weather_data.csv"
SITE_30MIN_DIR = CLEANED_WORKSPACE / "checkpoints" / "site_30min"
AUDIT_DIR = CLEANED_WORKSPACE / "audits"
VALID_SITES_PATH = AUDIT_DIR / "aedp_valid_checkpoint_sites.csv"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
ROOT_DATA_DIR = REPO_ROOT / "data"
PUBLICATION_DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results" / "03_aedp_aggregation_level"
DATA_EXPLORATION_DIR = RESULTS_DIR / "01_data_exploration"
for directory in [RESULTS_DIR, DATA_EXPLORATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

START = pd.Timestamp("2021-07-01 00:00:00")
END_30MIN = pd.Timestamp("2024-06-30 23:30:00")
INDEX_30MIN = pd.date_range(START, END_30MIN, freq="30min", name="datetime")
RANDOM_SEED = 20260521
SAMPLES_PER_LEVEL = 3
AGGREGATION_LEVELS = [1, 10, 100, 1000]
MIN_VALID_SITES = 100

for path in [SITE_LIST_PATH, WEATHER_PATH, SITE_30MIN_DIR, VALID_SITES_PATH, ROOT_DATA_DIR, PUBLICATION_DATA_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)
print(f"Cleaned AEDP workspace: {CLEANED_WORKSPACE}")
print(f"Valid checkpoint site list: {VALID_SITES_PATH}")


Publication project: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1
Repository root: C:\Users\z5404477\Documents\PyNNLF


Cleaned AEDP workspace: C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\3. cleaned\AEDP_different_aggregation
Valid checkpoint site list: C:\Users\z5404477\OneDrive - UNSW\H0424909\04_Workspace\2. WIP\data\3. cleaned\AEDP_different_aggregation\audits\aedp_valid_checkpoint_sites.csv


## 2. Load Valid Checkpoint Sites And Weather

This notebook samples only from the valid checkpoint sites produced by notebook `2.1`. Sites excluded during checkpointing are not sampled. The weather variables are the same three variables used by the existing AEDP weather dataset.


In [2]:
site_list = pd.read_excel(SITE_LIST_PATH)
site_list["edp_site_id"] = site_list["edp_site_id"].astype(str).str.strip()
valid_sites = pd.read_csv(VALID_SITES_PATH)
valid_sites["edp_site_id"] = valid_sites["edp_site_id"].astype(str).str.strip()
valid_sites = valid_sites.loc[valid_sites["status"].eq("valid")].copy()
site_ids = valid_sites["edp_site_id"].tolist()

if len(site_ids) != len(set(site_ids)):
    raise ValueError("Valid checkpoint site list contains duplicate site IDs")
if len(site_ids) < MIN_VALID_SITES:
    raise ValueError(f"Only {len(site_ids)} valid checkpoint sites are available; at least {MIN_VALID_SITES} are needed for 100hh sampling")

missing_checkpoint_files = [site_id for site_id in site_ids if not (SITE_30MIN_DIR / f"{site_id}_30min.parquet").exists()]
if missing_checkpoint_files:
    raise FileNotFoundError("Valid site list references missing checkpoint files. First missing IDs: " + str(missing_checkpoint_files[:10]))

site_metadata = valid_sites.merge(
    site_list[["edp_site_id", "postcode", "state", "lat_deg", "long_deg"]],
    on="edp_site_id",
    how="left",
    suffixes=("", "_site_list"),
)
weather = pd.read_csv(WEATHER_PATH, parse_dates=["datetime"]).set_index("datetime").reindex(INDEX_30MIN)
expected_weather_columns = ["air_temperature_in_degrees_c", "relative_humidity_in_percentage", "wind_speed_in_km_h"]
if list(weather.columns) != expected_weather_columns:
    raise ValueError(f"Unexpected weather columns: {list(weather.columns)}")
if weather.isna().any().any():
    raise ValueError("Weather data contains missing values after reindexing")

print(f"Valid checkpoint sites available for sampling: {len(site_ids)}")
display(site_metadata.head())
display(weather.head())


Valid checkpoint sites available for sampling: 139


,edp_site_id,raw_5min_points,status,first_raw_datetime,last_raw_datetime,native_missing_before_ffill,native_missing_after_ffill,checkpoint_rows,checkpoint_path,postcode,state,lat_deg,long_deg,date_of_first_data,date_of_last_data,number_of_days,postcode_site_list,state_site_list,lat_deg_site_list,long_deg_site_list
0,S0405,235299,valid,2021-07-01,2024-06-30 23:55:00,80349,0,52608,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,2282,NSW,-32.9725,151.6527,2021-06-28,2024-07-12,1110,2282,NSW,-32.9725,151.6527
1,W0013,311013,valid,2021-07-01,2024-06-30 13:55:00,4635,0,52608,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,2226,NSW,-33.9994,151.0639,2020-10-13,2024-07-12,1368,2226,NSW,-33.9994,151.0639
2,S0321,242174,valid,2021-07-01,2024-06-30 23:55:00,73474,0,52608,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,2322,NSW,-32.7931,151.6769,2021-06-03,2024-07-13,1136,2322,NSW,-32.7931,151.6769
3,S0431,314981,valid,2021-07-01,2024-06-30 23:55:00,667,0,52608,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,2284,NSW,-32.9806,151.6184,2021-05-27,2024-07-16,1146,2284,NSW,-32.9806,151.6184
4,S0170,302882,valid,2021-07-01,2024-06-30 23:55:00,12766,0,52608,C:\Users\z5404477\OneDrive - UNSW\H0424909\04_...,2307,NSW,-32.8788,151.6910,2021-03-08,2024-07-16,1226,2307,NSW,-32.8788,151.6910


,air_temperature_in_degrees_c,relative_humidity_in_percentage,wind_speed_in_km_h
datetime,,,
2021-07-01 00:00:00,13.7,96.0,13.0
2021-07-01 00:30:00,13.2,96.0,9.4
2021-07-01 01:00:00,13.3,97.0,11.2
2021-07-01 01:30:00,13.2,97.0,7.6
2021-07-01 02:00:00,12.9,98.0,7.6


## 3. Create Reproducible Sample Design And Display Chosen Households

This section creates 3 random samples per aggregation level:

- `1hh`: 3 single-household samples.
- `10hh`: 3 groups of 10 households, sampled without replacement within each group.
- `100hh`: 3 groups of 100 households, sampled without replacement within each group.
- `1000hh`: 3 synthetic bootstrap groups, sampled with replacement from the valid checkpoint sites.

Groups may overlap across samples. No duplicates are allowed within the 1/10/100hh groups. The 1000hh level intentionally allows repeated household IDs and stores those repetitions as integer weights.


In [3]:
rng = np.random.default_rng(RANDOM_SEED)
dataset_rows = []
membership_rows = []
next_dataset_no = 25
expected_dataset_count = SAMPLES_PER_LEVEL * len(AGGREGATION_LEVELS)
expected_dataset_ids = [f"ds{i}" for i in range(25, 25 + expected_dataset_count)]

for level in AGGREGATION_LEVELS:
    for sample_no in range(1, SAMPLES_PER_LEVEL + 1):
        dataset_id = f"ds{next_dataset_no}"
        if level < 1000:
            selected = rng.choice(site_ids, size=level, replace=False).tolist()
        else:
            selected = rng.choice(site_ids, size=level, replace=True).tolist()
        counts = pd.Series(selected, name="edp_site_id").value_counts().sort_index()
        for site_id, weight in counts.items():
            membership_rows.append({"dataset_id": dataset_id, "aggregation_level_hh": level, "sample_no": sample_no, "edp_site_id": site_id, "weight": int(weight)})
        dataset_rows.append({
            "dataset_id": dataset_id,
            "dataset_no": next_dataset_no,
            "aggregation_level_hh": level,
            "sample_no": sample_no,
            "sample_label": f"{level}hh_sample{sample_no:02d}",
            "filename": f"{dataset_id}_aedp_{level}hh_sample{sample_no:02d}_30min_with_weather.csv",
            "unique_households": int(counts.shape[0]),
            "total_household_weight": int(counts.sum()),
        })
        next_dataset_no += 1

sample_design = pd.DataFrame(dataset_rows)
membership = pd.DataFrame(membership_rows)
if sample_design.shape[0] != expected_dataset_count or sample_design["dataset_id"].tolist() != expected_dataset_ids:
    raise ValueError(f"Sample design should contain {expected_dataset_count} datasets: {expected_dataset_ids[0]} through {expected_dataset_ids[-1]}")

membership_display = membership.merge(
    site_metadata[["edp_site_id", "postcode", "state", "lat_deg", "long_deg"]],
    on="edp_site_id",
    how="left",
).sort_values(["dataset_id", "edp_site_id"])

sample_design.to_csv(AUDIT_DIR / "aedp_aggregation_sample_design.csv", index=False)
membership.to_csv(AUDIT_DIR / "aedp_aggregation_sample_membership.csv", index=False)
membership_display.to_csv(AUDIT_DIR / "aedp_aggregation_sample_membership_with_metadata.csv", index=False)
sample_design.to_csv(DATA_EXPLORATION_DIR / "aedp_aggregation_sample_design.csv", index=False)
membership.to_csv(DATA_EXPLORATION_DIR / "aedp_aggregation_sample_membership.csv", index=False)
membership_display.to_csv(DATA_EXPLORATION_DIR / "aedp_aggregation_sample_membership_with_metadata.csv", index=False)

display(sample_design)
display(membership_display)


,dataset_id,dataset_no,aggregation_level_hh,sample_no,sample_label,filename,unique_households,total_household_weight
0,ds25,25,1,1,1hh_sample01,ds25_aedp_1hh_sample01_30min_with_weather.csv,1,1
1,ds26,26,1,2,1hh_sample02,ds26_aedp_1hh_sample02_30min_with_weather.csv,1,1
2,ds27,27,1,3,1hh_sample03,ds27_aedp_1hh_sample03_30min_with_weather.csv,1,1
3,ds28,28,10,1,10hh_sample01,ds28_aedp_10hh_sample01_30min_with_weather.csv,10,10
4,ds29,29,10,2,10hh_sample02,ds29_aedp_10hh_sample02_30min_with_weather.csv,10,10
5,ds30,30,10,3,10hh_sample03,ds30_aedp_10hh_sample03_30min_with_weather.csv,10,10
6,ds31,31,100,1,100hh_sample01,ds31_aedp_100hh_sample01_30min_with_weather.csv,100,100
7,ds32,32,100,2,100hh_sample02,ds32_aedp_100hh_sample02_30min_with_weather.csv,100,100
8,ds33,33,100,3,100hh_sample03,ds33_aedp_100hh_sample03_30min_with_weather.csv,100,100
9,ds34,34,1000,1,1000hh_sample01,ds34_aedp_1000hh_sample01_30min_with_weather.csv,139,1000


,dataset_id,aggregation_level_hh,sample_no,edp_site_id,weight,postcode,state,lat_deg,long_deg
0,ds25,1,1,W0250,1,2080,NSW,-33.6420,151.1287
1,ds26,1,2,S0478,1,2322,NSW,-32.7931,151.6769
2,ds27,1,3,S0596,1,2203,NSW,-33.9041,151.1394
3,ds28,10,1,S0135,1,2321,NSW,-32.7908,151.5154
4,ds28,10,1,S0216,1,2289,NSW,-32.9428,151.6958
...,...,...,...,...,...,...,...,...,...
745,ds36,1000,3,W0302,9,2031,NSW,-33.9121,151.2588
746,ds36,1000,3,W0319,5,2153,NSW,-33.7588,150.9929
747,ds36,1000,3,W0324,4,2170,NSW,-33.9522,150.8995
748,ds36,1000,3,W0336,6,2065,NSW,-33.8257,151.1931


## 4. Forward-Fill Households, Then Aggregate And Export

For each dataset sample, the notebook loads each selected household checkpoint, reindexes it to the full 30-minute study grid, forward-fills that household series, multiplies by the household's bootstrap weight, and only then adds it to the group total.

This is intentional. It differs from the historical `ds10`/`ds11` validation in notebook 1, where the aggregate was summed at raw timestamp level before forward-fill. For this aggregation-level experiment, we want missingness handled at the household level before constructing 1hh, 10hh, 100hh, and bootstrap 1000hh groups.


In [4]:
def load_weighted_group_series(weights: pd.Series) -> pd.Series:
    total = pd.Series(0.0, index=INDEX_30MIN, name="netload_kW")
    for site_id, weight in weights.items():
        path = SITE_30MIN_DIR / f"{site_id}_30min.parquet"
        frame = pd.read_parquet(path)
        series = frame.set_index("datetime")["netload_kW"].reindex(INDEX_30MIN)

        # Intentional order for this experiment: fill each household before aggregation.
        series = series.ffill()
        if series.isna().any():
            raise ValueError(f"{site_id}: checkpoint still has missing values after per-household forward-fill")

        total = total.add(series * int(weight), fill_value=0.0)
    return total


export_rows = []
for row in sample_design.itertuples(index=False):
    member = membership.loc[membership["dataset_id"].eq(row.dataset_id)]
    weights = member.set_index("edp_site_id")["weight"]
    netload = load_weighted_group_series(weights)
    dataset = netload.to_frame().merge(weather, left_index=True, right_index=True, how="left").reset_index()
    if dataset.shape[0] != 52608 or dataset.isna().any().any() or dataset["datetime"].duplicated().any():
        raise ValueError(f"{row.dataset_id}: exported dataset failed validation")
    for target_dir in [ROOT_DATA_DIR, PUBLICATION_DATA_DIR]:
        out_path = target_dir / row.filename
        dataset.to_csv(out_path, index=False)
        print(f"Wrote: {out_path}")
    export_rows.append({
        "dataset_id": row.dataset_id,
        "filename": row.filename,
        "aggregation_level_hh": row.aggregation_level_hh,
        "sample_no": row.sample_no,
        "rows": dataset.shape[0],
        "start": dataset["datetime"].min(),
        "end": dataset["datetime"].max(),
        "unique_households": row.unique_households,
        "total_household_weight": row.total_household_weight,
        "fill_order": "per_household_forward_fill_before_aggregation",
    })

export_summary = pd.DataFrame(export_rows)
export_summary.to_csv(AUDIT_DIR / "aedp_aggregation_dataset_export_summary.csv", index=False)
export_summary.to_csv(DATA_EXPLORATION_DIR / "aedp_aggregation_dataset_export_summary.csv", index=False)
display(export_summary)


Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds25_aedp_1hh_sample01_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds25_aedp_1hh_sample01_30min_with_weather.csv


Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds26_aedp_1hh_sample02_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds26_aedp_1hh_sample02_30min_with_weather.csv


Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds27_aedp_1hh_sample03_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds27_aedp_1hh_sample03_30min_with_weather.csv


Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds28_aedp_10hh_sample01_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds28_aedp_10hh_sample01_30min_with_weather.csv


Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds29_aedp_10hh_sample02_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds29_aedp_10hh_sample02_30min_with_weather.csv


...

Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds33_aedp_100hh_sample03_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds33_aedp_100hh_sample03_30min_with_weather.csv


Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds34_aedp_1000hh_sample01_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds34_aedp_1000hh_sample01_30min_with_weather.csv


Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds35_aedp_1000hh_sample02_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds35_aedp_1000hh_sample02_30min_with_weather.csv


Wrote: C:\Users\z5404477\Documents\PyNNLF\data\ds36_aedp_1000hh_sample03_30min_with_weather.csv
Wrote: C:\Users\z5404477\Documents\PyNNLF\publication\journal_article_1\data\ds36_aedp_1000hh_sample03_30min_with_weather.csv


,dataset_id,filename,aggregation_level_hh,sample_no,rows,start,end,unique_households,total_household_weight,fill_order
0,ds25,ds25_aedp_1hh_sample01_30min_with_weather.csv,1,1,52608,2021-07-01,2024-06-30 23:30:00,1,1,per_household_forward_fill_before_aggregation
1,ds26,ds26_aedp_1hh_sample02_30min_with_weather.csv,1,2,52608,2021-07-01,2024-06-30 23:30:00,1,1,per_household_forward_fill_before_aggregation
2,ds27,ds27_aedp_1hh_sample03_30min_with_weather.csv,1,3,52608,2021-07-01,2024-06-30 23:30:00,1,1,per_household_forward_fill_before_aggregation
3,ds28,ds28_aedp_10hh_sample01_30min_with_weather.csv,10,1,52608,2021-07-01,2024-06-30 23:30:00,10,10,per_household_forward_fill_before_aggregation
4,ds29,ds29_aedp_10hh_sample02_30min_with_weather.csv,10,2,52608,2021-07-01,2024-06-30 23:30:00,10,10,per_household_forward_fill_before_aggregation
5,ds30,ds30_aedp_10hh_sample03_30min_with_weather.csv,10,3,52608,2021-07-01,2024-06-30 23:30:00,10,10,per_household_forward_fill_before_aggregation
6,ds31,ds31_aedp_100hh_sample01_30min_with_weather.csv,100,1,52608,2021-07-01,2024-06-30 23:30:00,100,100,per_household_forward_fill_before_aggregation
7,ds32,ds32_aedp_100hh_sample02_30min_with_weather.csv,100,2,52608,2021-07-01,2024-06-30 23:30:00,100,100,per_household_forward_fill_before_aggregation
8,ds33,ds33_aedp_100hh_sample03_30min_with_weather.csv,100,3,52608,2021-07-01,2024-06-30 23:30:00,100,100,per_household_forward_fill_before_aggregation
9,ds34,ds34_aedp_1000hh_sample01_30min_with_weather.csv,1000,1,52608,2021-07-01,2024-06-30 23:30:00,139,1000,per_household_forward_fill_before_aggregation


## 5. Debug Notes

If checkpoint files are missing, run notebook `2.1_build_aedp_site_30min_checkpoints.ipynb` first. Re-running this notebook overwrites CSV outputs deterministically because the random seed and sample count are fixed.

Expected outputs are 12 datasets: `ds25` through `ds36`. The chosen household membership is displayed above and saved as `aedp_aggregation_sample_membership_with_metadata.csv`.
